# 5. Kreiranje dimenzijskog modela (Star shema)

Skripta za generiranje dimenzijskog modela podataka → star schema.

Dimenzijski model podataka je zvjezdasti model koji se sastoji od jedne tablice činjenica
i više tablica dimenzija (data mart). Ovom skriptom stvaramo shemu.

### Dimenzije:
- `dim_projekt` — projekt na kojem je ticket otvoren
- `dim_tehnicar` — tehničari (reporter i assignee)
- `dim_prioritet` — razina prioriteta ticketa
- `dim_status` — status ticketa
- `dim_vrijeme` — vremenska dimenzija

### Fact tablica:
- `fact_support_tickets` — sadrži metrike i strane ključeve prema dimenzijama

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, Column, Integer, BigInteger, String, Date, Float, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
engine = create_engine(DATABASE_URL, echo=False)
Session = sessionmaker(bind=engine)
session = Session()
Base = declarative_base()

print(f"Spojeno na bazu: {DB_NAME} ({DB_HOST})")

Spojeno na bazu: fipu_srp_projekt (localhost)


C:\Users\ipavl\AppData\Local\Temp\ipykernel_37728\2259122783.py:21: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


## 5.1 Definicija dimenzijskih tablica (ORM klase)

In [2]:
class DimProjekt(Base):
    """Dimenzija projekata — 15 unikatnih projekata."""
    __tablename__ = 'dim_projekt'

    projekt_key = Column(Integer, primary_key=True, autoincrement=True)
    naziv_projekta = Column(String(255))


class DimTehnicar(Base):
    """Dimenzija tehničara — reporteri i assigneei (100 osoba)."""
    __tablename__ = 'dim_tehnicar'

    tehnicar_key = Column(Integer, primary_key=True, autoincrement=True)
    ime_prezime = Column(String(255))


class DimPrioritet(Base):
    """Dimenzija prioriteta — 7 razina (Blocker, Highest, High, Medium, Low, Lowest, unknown)."""
    __tablename__ = 'dim_prioritet'

    prioritet_key = Column(Integer, primary_key=True, autoincrement=True)
    razina_prioriteta = Column(String(50))


class DimStatus(Base):
    """Dimenzija statusa — 15 mogućih statusa (closed, done, waiting, open, ...)."""
    __tablename__ = 'dim_status'

    status_key = Column(Integer, primary_key=True, autoincrement=True)
    naziv_statusa = Column(String(50))


class DimVrijeme(Base):
    """Vremenska dimenzija — jedan redak po datumu kreiranja ticketa."""
    __tablename__ = 'dim_vrijeme'

    vrijeme_key = Column(Date, primary_key=True)
    dan = Column(Integer)
    mjesec = Column(Integer)
    godina = Column(Integer)
    kvartal = Column(Integer)
    dan_u_tjednu = Column(String(20))


print("Dimenzijske klase definirane: DimProjekt, DimTehnicar, DimPrioritet, DimStatus, DimVrijeme")

Dimenzijske klase definirane: DimProjekt, DimTehnicar, DimPrioritet, DimStatus, DimVrijeme


## 5.2 Definicija tablice činjenica (Fact Table)

In [3]:
class FactSupportTickets(Base):
    """
    Tablica činjenica — jedan redak = jedan helpdesk ticket.
    
    Grain: jedan ticket.
    
    FK ključevi:
        - projekt_key → dim_projekt
        - reporter_key → dim_tehnicar (tko je prijavio)
        - assignee_key → dim_tehnicar (tko rješava, nullable — 46% ticketa nema assigneea)
        - prioritet_key → dim_prioritet
        - status_key → dim_status
        - vrijeme_key → dim_vrijeme
    
    Metrike:
        - vrijeme_rjesavanja_sati — ukupno vrijeme od kreiranja do rezolucije (sati)
        - broj_komentara — broj komentara na ticketu
        - sati_open — vrijeme provedeno u stanju 'open' (sati)
        - sati_in_progress — vrijeme u stanju 'in_progress' (sati)
        - sati_resolved — vrijeme u stanju 'resolved' (sati)
        - sati_waiting — vrijeme u stanju 'waiting' (sati)
    """
    __tablename__ = 'fact_support_tickets'

    ticket_id = Column(Integer, primary_key=True)

    # Strani ključevi prema dimenzijama
    projekt_key = Column(Integer, ForeignKey('dim_projekt.projekt_key'))
    reporter_key = Column(Integer, ForeignKey('dim_tehnicar.tehnicar_key'))
    assignee_key = Column(Integer, ForeignKey('dim_tehnicar.tehnicar_key'))
    prioritet_key = Column(Integer, ForeignKey('dim_prioritet.prioritet_key'))
    status_key = Column(Integer, ForeignKey('dim_status.status_key'))
    vrijeme_key = Column(Date, ForeignKey('dim_vrijeme.vrijeme_key'))

    # Metrike
    vrijeme_rjesavanja_sati = Column(Float)
    broj_komentara = Column(Integer)
    sati_open = Column(Float)
    sati_in_progress = Column(Float)
    sati_resolved = Column(Float)
    sati_waiting = Column(Float)


print("Fact klasa definirana: FactSupportTickets")

Fact klasa definirana: FactSupportTickets


## 5.3 Kreiranje tablica u bazi

In [4]:
from sqlalchemy import text

# Prvo dropaj stare tablice (redoslijed: fact -> dimenzije zbog FK)
drop_statements = [
    "DROP TABLE IF EXISTS fact_support_tickets",
    "DROP TABLE IF EXISTS dim_vrijeme",
    "DROP TABLE IF EXISTS dim_projekt",
    "DROP TABLE IF EXISTS dim_tehnicar",
    "DROP TABLE IF EXISTS dim_prioritet_status",
    "DROP TABLE IF EXISTS dim_prioritet",
    "DROP TABLE IF EXISTS dim_status",
]

with engine.connect() as conn:
    for stmt in drop_statements:
        conn.execute(text(stmt))
    conn.commit()
    print("Stare tablice obrisane.")

# Kreiraj nove tablice iz ORM definicija
Base.metadata.create_all(engine)
print("Sve tablice dimenzijskog modela su uspješno kreirane!")

Stare tablice obrisane.
Sve tablice dimenzijskog modela su uspješno kreirane!


## 5.4 Verifikacija kreiranih tablica

In [5]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("Kreirane tablice:")
for tbl in tables:
    cols = inspector.get_columns(tbl)
    col_names = [c['name'] for c in cols]
    fks = inspector.get_foreign_keys(tbl)
    fk_count = len(fks)
    print(f"  {tbl:30s} {len(cols)} stupaca, {fk_count} FK  →  {col_names}")

Kreirane tablice:
  dim_prioritet                  2 stupaca, 0 FK  →  ['prioritet_key', 'razina_prioriteta']
  dim_projekt                    2 stupaca, 0 FK  →  ['projekt_key', 'naziv_projekta']
  dim_status                     2 stupaca, 0 FK  →  ['status_key', 'naziv_statusa']
  dim_tehnicar                   2 stupaca, 0 FK  →  ['tehnicar_key', 'ime_prezime']
  dim_vrijeme                    6 stupaca, 0 FK  →  ['vrijeme_key', 'dan', 'mjesec', 'godina', 'kvartal', 'dan_u_tjednu']
  fact_support_tickets           13 stupaca, 6 FK  →  ['ticket_id', 'projekt_key', 'reporter_key', 'assignee_key', 'prioritet_key', 'status_key', 'vrijeme_key', 'vrijeme_rjesavanja_sati', 'broj_komentara', 'sati_open', 'sati_in_progress', 'sati_resolved', 'sati_waiting']
  support_tickets                42 stupaca, 0 FK  →  ['id', 'started', 'ended', 'issue_num', 'issue_proj', 'issue_reporter', 'issue_assignee', 'issue_contr_count', 'issue_type', 'issue_priority', 'issue_created', 'issue_resolution_date',